# DEMO 3: Uploading Files via the Databricks UI

In this demo you will learn two ways to get your own files into Databricks using the UI:

1. **Upload a CSV to create a table** 
2. **Upload a file to a Volume** — store files in a governed location without immediately creating a table

You will use a sample sales CSV file for both exercises.

---

## Step 1: Run Setup

The cell below creates a schema and a Volume in your workspace that you will upload files into.

Click **Run** (▶) on the cell below.

In [0]:
%run ./demo_3_setup

---

## Step 2: Download the Sample CSV

Before you can practice uploading, you need a file on your **local computer**. A sample CSV file (`sample_sales_data.csv`) has been placed in your workspace.

### How to download it:

1. In the **left sidebar**, click the **Workspace** icon (folder icon)
2. Navigate to your home folder (it shows your email address)
3. Find the file **`sample_sales_data.csv`**
4. Right-click on the file (or click the **⋮** menu next to it) and select **Download**
5. Save it somewhere on your computer where you can find it easily (e.g., your Desktop)

Once downloaded, you’re ready to upload it back through the Databricks UI — simulating the real-world scenario of "I have a CSV on my laptop that I want to get into Databricks."

---

## Step 3: Upload CSV to Create a Table

This is the simplest way to bring data into Databricks

### Follow these steps:

1. Click **+ New** in the left sidebar (the blue button near the top)
2. Select **Add or upload data**
3. Click **Create or modify a table**
4. **Drag and drop** your `sample_sales_data.csv` file onto the upload area (or click "browse" to find it)
5. Wait for the file to upload — you’ll see a preview of the data

### Configure the table:

6. Under **Catalog**, select your workspace catalog (the one shown in the setup output above)
7. Under **Schema**, find and select the schema starting with `demo_uploads_` followed by your username
8. The **Table name** will default to `sample_sales_data` — leave it as-is
9. Review the **data preview**:
   - Check that column names look correct (order_date, product_name, category, etc.)
   - Check that data types were detected (dates, strings, numbers)
   - You can click a column type icon to change it if needed
10. Click **Create table** at the bottom

### What just happened:

- Databricks created a **managed Delta Lake table** in Unity Catalog
- The data is stored in optimised Delta format (not as a raw CSV)
- The table is immediately queryable by anyone with `SELECT` permission
- It appears in Catalog Explorer and is governed like any other table

> **Limits:** You can upload up to 10 files at once, with a total size under 2 GB. Supported formats: CSV, TSV, JSON, Avro, Parquet, and text files.

---

## Step 4: Verify Your Uploaded Table

Run the cells below to confirm the table was created successfully. If you used a different table name in Step 3, update it below.

In [0]:
%sql
-- Check that the table exists in your demo schema
SHOW TABLES;

In [0]:
%sql
-- Query the uploaded table
SELECT * FROM sample_sales_data;

In [0]:
%sql
-- Check the table details - notice it's a MANAGED Delta table
DESCRIBE TABLE EXTENDED sample_sales_data;

Your CSV file is now a **governed Delta Lake table** in Unity Catalog. This table is accessible from any notebook, any SQL query, any dashboard, and any user who has been granted access.

---

## Step 5: Upload a File to a Volume

Sometimes you don’t want to create a table immediately — you just want to store a file in a governed location. That’s what **Volumes** are for.

Think of a Volume as a governed folder in cloud storage — like OneDrive or SharePoint for your data files, but with Unity Catalog governance. Volume paths follow this pattern:

```
/Volumes/catalog/schema/volume_name/
```

### Follow these steps:

1. Click **+ New** in the left sidebar
2. Select **Add or upload data**
3. Click **Upload files to a volume**
4. **Drag and drop** your `sample_sales_data.csv` file onto the upload area
5. Under **Destination volume**, select:
   - Your workspace **catalog**
   - The schema starting with `demo_uploads_`
   - The volume called **`landing_files`**
6. Click **Upload**

### What just happened:

- The file was stored **as a file** in the Volume (not converted to a table)
- It’s accessible at the path `/Volumes/<catalog>/<schema>/landing_files/sample_sales_data.csv`
- You can later create tables from it, reference it in notebooks, or process it with AI functions
- The file is governed by Unity Catalog — access is controlled by Volume permissions

> **In Databricks, uploading to a Volume keeps the file accessible as a file — you can create multiple tables from it, reference it in different ways, or just use it as-is.

> **Limits:** Files up to 5 GB can be uploaded via the UI. For larger files, use the Python SDK.

---

## Step 6: Verify the File in the Volume

Let’s confirm the file is in the Volume and query it directly. The setup notebook already placed a copy there, but if you uploaded your own copy you’ll see it listed too.

In [0]:
# List files in the volume
import os

files = os.listdir(volume_path)
print(f"Files in {volume_path}:")
for f in files:
    size = os.path.getsize(f"{volume_path}/{f}")
    print(f"  \u2022 {f} ({size:,} bytes)")

In [0]:
%sql
# Query the CSV file directly from the Volume using read_files()
# This reads the file without creating a permanent table
file_path = f"{volume_path}/sample_sales_data.csv"
df = spark.sql(f"""
  SELECT * FROM read_files(
    '{file_path}',
    format => 'csv',
    header => true
  )
""")
display(df)

This is powerful: you can **query files directly from a Volume** without first creating a table. The `read_files()` function reads the CSV with automatic type detection.

You can also create a table from a Volume file at any time:

In [0]:
%sql
# Create a table from a file in a Volume (you don't need to re-upload!)
spark.sql(f"""
  CREATE OR REPLACE TABLE sales_from_volume
  AS SELECT * FROM read_files(
    '{volume_path}/sample_sales_data.csv',
    format => 'csv',
    header => true
  )
""")
print(f"\u2713 Table 'sales_from_volume' created from the Volume file")

In [0]:
%sql
-- Verify the table was created
SELECT COUNT(*) AS row_count FROM sales_from_volume;

---

## Summary

In this demo you have:

1. **Uploaded a CSV to create a table** — using the "Create or modify a table" wizard
2. **Uploaded a file to a Volume** — storing it as a governed file that can be queried directly or used to create tables later
3. **Queried a file directly from a Volume** — using `read_files()` to read CSV data without creating a permanent table
4. **Created a table from a Volume file** — showing that files in Volumes can be turned into tables at any time

| Method | When to Use |
|---|---|
| **Create or modify a table** | You want the data immediately available as a queryable table |
| **Upload to a Volume** | You want to store the file first and decide what to do with it later (or you have non-tabular files like PDFs, images, etc.) |

In [0]:
# Cleanup: uncomment and run the lines below ONLY if you want to remove the demo objects
# spark.sql(f"DROP SCHEMA IF EXISTS `{catalog_name}`.`{schema_name}` CASCADE")
# print("\u2713 Demo schema and volume removed")